In [1]:
import numpy as np  
import pandas as pd  
from sklearn.model_selection import train_test_split, GridSearchCV  
from sklearn.feature_extraction.text import TfidfVectorizer  
from keras.models import Sequential  
from keras.layers import Dense, LSTM, Embedding, SpatialDropout1D, Conv1D, MaxPooling1D  
from keras.utils import pad_sequences  
from keras.callbacks import EarlyStopping  
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score  
from keras.wrappers.scikit_learn import KerasClassifier 



In [2]:
df = pd.read_csv('nama_file.csv')

df.head()

,title,abstract,label
0,HUBUNGAN KECEPATAN LARI DENGAN KETERAMPILAN DR...,"Ria irawan laponangi. H11901. ""Hubungan Kecepa...",0
1,SURVEI TINGKAT KESEGARAN JASMANI SISWA KELAS X...,"Asri H119073 ""Survei Tingkat Kesegaran Jasmani...",1
2,HUBUNGAN KOORDINASI MATA KAKI DENGAN KETERAMPI...,"Hasanudin. H119029 ""Hubungan Koordinası Mata K...",1
3,KONTRIBUSI TINGKAT KESEGARAN JASMANI TERHADAP ...,"Andi Mallarangen, H119079 Kontribusi Tingkat K...",0
4,HUBUNGAN KEKUATAN OTOT LENGAN DENGAN KEMAMPUAN...,"Arifin. H119038 ""Hubungan kekuatan otot lengan...",1


In [3]:
X = df['title'] + " " + df['abstract']  
y = df['label']  # 1 untuk plagiasi, 0 untuk non-plagiasi  s

In [4]:
# Vectorize text  
vectorizer = TfidfVectorizer(max_features=10000)  
X_vectorized = vectorizer.fit_transform(X).toarray()  

# Split data  
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)  
 


In [7]:
# Define model builder function  
def build_model(units_lstm1=128, units_lstm2=64, dropout_rate=0.3, recurrent_dropout_rate=0.3, batch_size=64, epochs=20):  
    model = Sequential()  
    model.add(Embedding(input_dim=10000, output_dim=128, input_length=X_train.shape[1]))  
    model.add(SpatialDropout1D(0.2))  
    model.add(Conv1D(filters=64, kernel_size=5, padding='same', activation='relu'))  
    model.add(MaxPooling1D(pool_size=4))  
    model.add(LSTM(units_lstm1, dropout=dropout_rate, recurrent_dropout=recurrent_dropout_rate, return_sequences=True))  
    model.add(LSTM(units_lstm2, dropout=dropout_rate, recurrent_dropout=recurrent_dropout_rate))  
    model.add(Dense(1, activation='sigmoid'))  

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])  
    return model  

# Create KerasClassifier  
model = KerasClassifier(build_fn=build_model, verbose=2)  


C:\Users\USER\AppData\Local\Temp\ipykernel_7772\1986713281.py:16: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  model = KerasClassifier(build_fn=build_model, verbose=2)


In [ ]:
# Early stopping  
early_stopping = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)  


In [ ]:
# Hyperparameter tuning  
param_grid = {  
    'units_lstm1': [128, 256],  
    'units_lstm2': [64, 128],  
    'dropout_rate': [0.2, 0.3, 0.4],  
    'recurrent_dropout_rate': [0.2, 0.3, 0.4],  
    'batch_size': [32, 64, 128],  
    'epochs': [20, 30, 40]  
}  

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1)  

grid_search.fit(X_train, y_train, validation_data=(X_test, y_test), callbacks=[early_stopping]) 

In [ ]:
# Evaluate best model  
best_model = grid_search.best_estimator_  
loss, accuracy = best_model.score(X_test, y_test)  
print(f'Akurasi: {accuracy}')  

# Save the model  
best_model.model.save("model_plagiasi.h5")  
print("Model telah disimpan sebagai 'model_plagiasi.h5'.")  

In [26]:
# Fungsi prediksi untuk similarity
def predict_similarity(title, abstract):
    title_seq = tokenizer.texts_to_sequences([title])
    abstract_seq = tokenizer.texts_to_sequences([abstract])
    
    max_title_length = 100  
    max_abstract_length = max_length  
    
    padded_title = pad_sequences(title_seq, maxlen=max_title_length, padding='post')
    padded_abstract = pad_sequences(abstract_seq, maxlen=max_abstract_length, padding='post')
    
    input_data = np.concatenate((padded_title, padded_abstract), axis=1)
    prediction = model.predict(input_data)
    return prediction[0][0]

# Contoh prediksi
predicted_similarity = predict_similarity(
    'Lorem ipsum is placeholder text commonly used in the graphic, print, and publishing industries for previewing layouts and visual mockups.',
    'Lorem ipsum is placeholder text commonly used in the graphic, print, and publishing industries for previewing layouts and visual mockups.'
)
print(f'Predicted similarity: {predicted_similarity}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 341ms/step
Predicted similarity: 1.0
